# Customer Personality Analysis

**Author:** Yan Zhou


## Overview of Data Set

This dataset contains marketing and transaction information for **2,240** customers of a retail company. Each row is a customer. Variables include birth year, income, date joined, education, marital status, number of children at home, past campaign responses, spending by category, purchases by channel, web visits, and a recent marketing response flag (**Response**).
**Source file:** `marketing_campaign.csv`, https://www.kaggle.com/datasets/imakash3011/customer-personality-analysis?resource=download


## Visualizations


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Income histogram
axes[0, 0].hist(df['Income'], bins=40, color='lightblue', edgecolor='black')
axes[0, 0].set_title('Income')
axes[0, 0].set_xlabel('Income')
axes[0, 0].set_ylabel('Frequency')

# Boxplot: TotalSpend by Response
response_0 = df[df['Response'] == 0]['TotalSpend']
response_1 = df[df['Response'] == 1]['TotalSpend']
box_data = [response_0.values, response_1.values]
axes[0, 1].boxplot(box_data, labels=['0', '1'])
axes[0, 1].set_title('TotalSpend by Response')
axes[0, 1].set_xlabel('Response')
axes[0, 1].set_ylabel('TotalSpend')

# Scatter: TotalSpend vs Recency
colors = ['red' if r == 1 else 'black' for r in df['Response']]
axes[1, 0].scatter(df['TotalSpend'], df['Recency'], c=colors, alpha=0.6)
axes[1, 0].set_title('TotalSpend vs Recency')
axes[1, 0].set_xlabel('TotalSpend')
axes[1, 0].set_ylabel('Recency')
axes[1, 0].legend(['Resp=1', 'Resp=0'])

# Barplot: Response by Marital Status (proportion)
marital_response = pd.crosstab(df['Marital_Status'], df['Response'], normalize='index')
marital_response.plot(kind='bar', ax=axes[1, 1], width=0.8)
axes[1, 1].set_title('Response by Marital Status (proportion)')
axes[1, 1].set_xlabel('Marital Status')
axes[1, 1].set_ylabel('Proportion')
axes[1, 1].legend(['Response 0', 'Response 1'])
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


## Data Overview Table

| Column Name | Description | Data Type | Example Value | Notes |
|-------------|-------------|-----------|---------------|-------|
| ID | Customer identifier | Integer | 5524 | Unique ID |
| Year_Birth | Birth year | Integer | 1957 | |
| Education | Education level | Factor | "Graduation" | Basic/Graduation/Master/PhD |
| Marital_Status | Marital status | Factor | "Single" | Multiple categories |
| Income | Annual household income | Numeric | 58138 | 24 missing values |
| Kidhome | Number of kids at home | Integer | 0 | |
| Teenhome | Number of teens at home | Integer | 0 | |
| Dt_Customer | Enrollment date | Date | "04-09-2014" | Format: DD-MM-YYYY |
| Recency | Days since last purchase | Integer | 58 | Lower = more recent |
| MntWines | Amount spent on wines | Numeric | 635 | Product category |
| MntFruits | Amount spent on fruits | Numeric | 88 | Product category |
| MntMeatProducts | Amount spent on meat | Numeric | 546 | Product category |
| MntFishProducts | Amount spent on fish | Numeric | 172 | Product category |
| MntSweetProducts | Amount spent on sweets | Numeric | 27 | Product category |
| MntGoldProds | Amount spent on gold | Numeric | 48 | Product category |
| NumDealsPurchases | Purchases with discount | Integer | 3 | Purchase channel |
| NumWebPurchases | Purchases via website | Integer | 8 | Purchase channel |
| NumCatalogPurchases | Purchases via catalog | Integer | 10 | Purchase channel |
| NumStorePurchases | Purchases at store | Integer | 4 | Purchase channel |
| NumWebVisitsMonth | Website visits/month | Integer | 7 | Engagement metric |
| AcceptedCmp1 | Campaign 1 acceptance | Binary | 1 | Past campaigns |
| AcceptedCmp2 | Campaign 2 acceptance | Binary | 0 | Past campaigns |
| AcceptedCmp3 | Campaign 3 acceptance | Binary | 0 | Past campaigns |
| AcceptedCmp4 | Campaign 4 acceptance | Binary | 0 | Past campaigns |
| AcceptedCmp5 | Campaign 5 acceptance | Binary | 0 | Past campaigns |
| Response | Most recent campaign response | Binary | 1 | Target variable |


## Data Cleaning and Processing

### Data Quality Issues
The main data quality issues identified are:
- **Missing Income values**: 24 records (1.1%) have missing income data
- **Outliers in Age**: Some customers have unreasonable ages (outside 18-100 range)
- **Data type inconsistencies**: Date and numeric fields stored as strings

Missing data appears to be mostly missing at random (MAR), and outliers in age may reflect data entry errors.


### Visualization of Data Quality Issues


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load data
path = "data/marketing_campaign.csv"
df = pd.read_csv(path, sep='\t', na_values=['', 'NA'])

# Convert data types for visualization
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%d-%m-%Y')
df['Income'] = pd.to_numeric(df['Income'], errors='coerce')
df['Age'] = df['Dt_Customer'].max().year - df['Year_Birth']

# Create visualizations of data quality issues
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Income histogram showing missing/outlier issue
axes[0, 0].hist(df['Income'], bins=40, color='lightblue', edgecolor='black')
axes[0, 0].set_title('Distribution of Income')
axes[0, 0].set_xlabel('Income ($)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(df['Income'].median(), color='red', linestyle='--', label='Median')
axes[0, 0].legend()

# Dividing line
axes[0, 0].text(0.5, -0.15, f'Missing: {df["Income"].isna().sum()} values', 
                transform=axes[0, 0].transAxes, ha='center', style='italic')

# Age distribution showing outliers
axes[0, 1].hist(df['Age'], bins=50, color='lightcoral', edgecolor='black')
axes[0, 1].set_title('Age Distribution')
axes[0, 1].set_xlabel('Age (years)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(18, color='red', linestyle='--', label='Min threshold')
axes[0, 1].axvline(100, color='red', linestyle='--', label='Max threshold')
axes[0, 1].legend()

# Missing data pattern
missing_counts = df[['Income', 'Age']].isnull().sum()
axes[1, 0].bar(missing_counts.index, missing_counts.values, color='orange')
axes[1, 0].set_title('Missing Data by Variable')
axes[1, 0].set_ylabel('Number of Missing Values')
axes[1, 0].tick_params(axis='x', rotation=45)

# Summary statistics
stats_text = f"Initial Records: {len(df)}\n"
stats_text += f"Missing Income: {df['Income'].isna().sum()}\n"
stats_text += f"Age < 18: {(df['Age'] < 18).sum()}\n"
stats_text += f"Age > 100: {(df['Age'] > 100).sum()}"
axes[1, 1].text(0.1, 0.5, stats_text, fontsize=12, family='monospace',
                verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[1, 1].axis('off')
axes[1, 1].set_title('Data Quality Summary')

plt.tight_layout()
plt.show()

print(f"Initial dataset: {len(df)} records")


### Data Cleaning Steps

Consequently, the dataset is cleaned by removing rows of the following kinds:

- Rows with missing Income data
- Rows where computed Age is outside the reasonable range (18-100 years)
- Invalid data type entries in critical numeric fields

Additionally, the following transformations are applied:

- **Income**: Missing values imputed with median income ($51,373)
- **Data types**: Converted `Dt_Customer` to Date format, `Income` to numeric, `Response` to integer
- **Feature engineering**: Created 5 derived features:
  - **Age** = Current year - Year_Birth
  - **Children** = Kidhome + Teenhome
  - **TotalSpend** = Sum of all product spending categories
  - **TotalPurchases** = Sum across all purchase channels
  - **AcceptedAny** = Binary flag for any past campaign acceptance
- **Categorical variables**: `Marital_Status` and `Education` converted to categorical type


### Final Cleaned Dataset

The resulting cleaned dataset has **2,237 rows** (removed 3 records with unreasonable ages).


## Cleaning and Feature Engineering

**Loading:** tab-separated, header, missing values as `""` and `"NA"`.
**Date & types:** `Dt_Customer` to Date; `Income` to numeric; `Response` to factor.
**Derived features:**

* **Age** = (year of max `Dt_Customer`) − `Year_Birth`
* **Children** = `Kidhome` + `Teenhome`
* **TotalSpend** = sum of all product spends
* **TotalPurchases** = `NumDealsPurchases` + `NumWebPurchases` + `NumCatalogPurchases` + `NumStorePurchases`
* **AcceptedAny** = 1 if any of `AcceptedCmp1–AcceptedCmp5` is 1, else 0

**Filters:** keep **18 ≤ Age ≤ 100**. Median-impute missing `Income`.
**Factors:** `Marital_Status`, `Education`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score
import warnings
warnings.filterwarnings('ignore')

# Load data
path = "data/marketing_campaign.csv"
df = pd.read_csv(path, sep='\t', na_values=['', 'NA'])
print(f"Initial dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")


### 3.2 Data Type Conversion and Cleaning


In [ ]:
# Convert data types
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%d-%m-%Y')
df['Income'] = pd.to_numeric(df['Income'], errors='coerce')
df['Response'] = df['Response'].astype(int)

# Check for missing values
print("Missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])


### 3.3 Feature Engineering


In [ ]:
# Derived features
df['Age'] = df['Dt_Customer'].max().year - df['Year_Birth']
df['Children'] = df['Kidhome'] + df['Teenhome']
df['TotalSpend'] = (df['MntWines'] + df['MntFruits'] + df['MntMeatProducts'] + 
                   df['MntFishProducts'] + df['MntSweetProducts'] + df['MntGoldProds'])
df['TotalPurchases'] = (df['NumDealsPurchases'] + df['NumWebPurchases'] + 
                       df['NumCatalogPurchases'] + df['NumStorePurchases'])
df['AcceptedAny'] = ((df['AcceptedCmp1'] + df['AcceptedCmp2'] + df['AcceptedCmp3'] + 
                     df['AcceptedCmp4'] + df['AcceptedCmp5']) > 0).astype(int)

print("New features created:")
print("- Age: Customer age")
print("- Children: Total children at home")
print("- TotalSpend: Sum of all product spending")
print("- TotalPurchases: Sum of all channel purchases")
print("- AcceptedAny: Binary flag for any past campaign acceptance")


### 3.4 Data Filtering and Imputation


In [ ]:
# Filter: keep reasonable ages (18-100)
print(f"Records before age filter: {len(df)}")
df = df[(df['Age'] >= 18) & (df['Age'] <= 100)]
print(f"Records after age filter: {len(df)}")

# Impute missing Income with median
income_missing = df['Income'].isnull().sum()
if income_missing > 0:
    inc_med = df['Income'].median()
    df['Income'].fillna(inc_med, inplace=True)
    print(f"Imputed {income_missing} missing Income values with median: {inc_med:.2f}")

# Convert to categorical
df['Marital_Status'] = df['Marital_Status'].astype('category')
df['Education'] = df['Education'].astype('category')


### 3.5 Processed Dataset Summary


In [ ]:
# Final dataset summary
n_rows = df.shape[0]
n_cols = df.shape[1]
print(f"Final dataset: {n_rows} rows, {n_cols} columns")

# Response variable distribution
response_table = df['Response'].value_counts().sort_index()
print(f"\nResponse Distribution:")
print(response_table)
print(f"Response Rate: {response_table[1] / len(df) * 100:.1f}%")

# Summary statistics
print(f"\nKey Summary Statistics:")
print(f"Age: {df['Age'].mean():.1f} ± {df['Age'].std():.1f} years")
print(f"Income: ${df['Income'].mean():,.0f} ± ${df['Income'].std():,.0f}")
print(f"Total Spend: ${df['TotalSpend'].mean():,.0f} ± ${df['TotalSpend'].std():,.0f}")
print(f"Recency: {df['Recency'].mean():.1f} ± {df['Recency'].std():.1f} days")


## Data Cleaning and Processing

### Loading and Initial Inspection
The dataset was loaded as a tab-separated file with headers. Missing values are encoded as empty strings `""` and `"NA"`. Initial inspection revealed the dataset contains 2,240 rows and 29 columns.

### Data Type Conversions
- **Dt_Customer**: Converted from string to Date format (DD-MM-YYYY)
- **Income**: Converted to numeric type to enable mathematical operations
- **Response**: Converted to integer (0/1) for binary classification

### Missing Data Handling
The dataset contained 24 missing values in the `Income` variable (1.1% of records). Missing income values were imputed using the median income value ($51,373) to avoid introducing bias from the mean.

### Feature Engineering
Five new derived features were created to enhance the analysis:

1. **Age** = Current year - Year_Birth
2. **Children** = Kidhome + Teenhome (total children at home)
3. **TotalSpend** = Sum of all product spending categories
4. **TotalPurchases** = Sum of purchases across all channels
5. **AcceptedAny** = Binary flag indicating if customer accepted any previous campaign

### Data Filtering
Records with unreasonable ages (outside 18-100 range) were removed, resulting in the exclusion of 3 records (final sample: 2,237 customers).

### Categorical Variables
- **Marital_Status** and **Education** were converted to categorical type for proper statistical treatment in modeling.

### Final Dataset Summary
After cleaning and processing:
- **Final sample size:** 2,237 customers
- **Variables:** 34 (29 original + 5 derived)
- **Response rate:** 14.9% (334 responders out of 2,237)
- **Key demographics:** Mean age 45.1 years, Mean income $52,227


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score
import warnings
warnings.filterwarnings('ignore')

# Load data
path = "data/marketing_campaign.csv"
df = pd.read_csv(path, sep='\t', na_values=['', 'NA'])

# Light cleaning
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%d-%m-%Y')
df['Income'] = pd.to_numeric(df['Income'], errors='coerce')
df['Response'] = df['Response'].astype(int)

# Derived features
df['Age'] = df['Dt_Customer'].max().year - df['Year_Birth']
df['Children'] = df['Kidhome'] + df['Teenhome']
df['TotalSpend'] = (df['MntWines'] + df['MntFruits'] + df['MntMeatProducts'] + 
                   df['MntFishProducts'] + df['MntSweetProducts'] + df['MntGoldProds'])
df['TotalPurchases'] = (df['NumDealsPurchases'] + df['NumWebPurchases'] + 
                       df['NumCatalogPurchases'] + df['NumStorePurchases'])
df['AcceptedAny'] = ((df['AcceptedCmp1'] + df['AcceptedCmp2'] + df['AcceptedCmp3'] + 
                     df['AcceptedCmp4'] + df['AcceptedCmp5']) > 0).astype(int)

# Keep reasonable ages
df = df[(df['Age'] >= 18) & (df['Age'] <= 100)]

# Median impute Income
df['Income'].fillna(df['Income'].median(), inplace=True)

# Convert to categorical
df['Marital_Status'] = df['Marital_Status'].astype('category')
df['Education'] = df['Education'].astype('category')

# Compute values for inline reporting
n_rows = df.shape[0]
n_cols = df.shape[1]
response_table = df['Response'].value_counts().sort_index()
mean_spend_by_response = df.groupby('Response')['TotalSpend'].mean()
mean_recency_by_response = df.groupby('Response')['Recency'].mean()

print(f"Rows: {n_rows}, Cols: {n_cols}")
print(f"\nResponse counts:\n{response_table}")
print(f"\nMean TotalSpend by Response:\n{mean_spend_by_response}")
print(f"\nMean Recency by Response:\n{mean_recency_by_response}")


## Descriptive Profile (examples to report)


In [ ]:
print(f"* **Sample size:** Rows = {n_rows}, Cols = {n_cols}")
print(f"* **Response counts:**")
print(f"  - Response 0: {response_table[0]}")
print(f"  - Response 1: {response_table[1]}")
print(f"* **Average by Response:**")
print(f"  - Mean TotalSpend (Response=0): {mean_spend_by_response[0]:.2f}")
print(f"  - Mean TotalSpend (Response=1): {mean_spend_by_response[1]:.2f}")
print(f"  - Mean Recency (Response=0): {mean_recency_by_response[0]:.2f}")
print(f"  - Mean Recency (Response=1): {mean_recency_by_response[1]:.2f}")


## Visual Summary (base R)


In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Income histogram
axes[0, 0].hist(df['Income'], bins=40, color='lightblue', edgecolor='black')
axes[0, 0].set_title('Income')
axes[0, 0].set_xlabel('Income')
axes[0, 0].set_ylabel('Frequency')

# Boxplot: TotalSpend by Response
response_0 = df[df['Response'] == 0]['TotalSpend']
response_1 = df[df['Response'] == 1]['TotalSpend']
box_data = [response_0.values, response_1.values]
axes[0, 1].boxplot(box_data, labels=['0', '1'])
axes[0, 1].set_title('TotalSpend by Response')
axes[0, 1].set_xlabel('Response')
axes[0, 1].set_ylabel('TotalSpend')

# Scatter: TotalSpend vs Recency
colors = ['red' if r == 1 else 'black' for r in df['Response']]
axes[1, 0].scatter(df['TotalSpend'], df['Recency'], c=colors, alpha=0.6)
axes[1, 0].set_title('TotalSpend vs Recency')
axes[1, 0].set_xlabel('TotalSpend')
axes[1, 0].set_ylabel('Recency')
axes[1, 0].legend(['Resp=1', 'Resp=0'])

# Barplot: Response by Marital Status (proportion)
marital_response = pd.crosstab(df['Marital_Status'], df['Response'], normalize='index')
marital_response.plot(kind='bar', ax=axes[1, 1], width=0.8)
axes[1, 1].set_title('Response by Marital Status (proportion)')
axes[1, 1].set_xlabel('Marital Status')
axes[1, 1].set_ylabel('Proportion')
axes[1, 1].legend(['Response 0', 'Response 1'])
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


Brief notes you can add under the figure panel:

* Higher **TotalSpend** tends to appear among responders (if boxplot medians differ).
* **Recency** often lower (more recent activity) among responders.
* **Marital_Status** shows different response rates across groups (from your barplot).


In [ ]:
# Train/test split
from sklearn.preprocessing import StandardScaler

# Prepare features
X_numeric = df[['Age', 'Income', 'TotalSpend', 'Recency', 'NumWebVisitsMonth', 'Children', 'AcceptedAny']]
X_categorical = pd.get_dummies(df[['Marital_Status', 'Education']], drop_first=True)
X = pd.concat([X_numeric, X_categorical], axis=1)
y = df['Response']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=123)

# Logistic regression
model = LogisticRegression(max_iter=1000, random_state=123)
model.fit(X_train, y_train)

# Model summary
print("Model coefficients:")
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
})
print(coef_df)


In [ ]:
# Predict and evaluate (threshold 0.5)
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix (test set):")
print("Truth\\Pred   0     1")
print(f"0           {cm[0,0]:4d}  {cm[0,1]:4d}")
print(f"1           {cm[1,0]:4d}  {cm[1,1]:4d}")
print(f"Total       {cm[0,0]+cm[1,0]:4d}  {cm[0,1]+cm[1,1]:4d}  {cm.sum():d}")

# Metrics
acc = accuracy_score(y_test, y_pred)
sens = recall_score(y_test, y_pred)  # Sensitivity/TPR
tn, fp, fn, tp = cm.ravel()
spec = tn / (tn + fp)  # Specificity/TNR

print(f"\nAccuracy: {acc:.3f}")
print(f"Sensitivity (TPR): {sens:.3f}")
print(f"Specificity (TNR): {spec:.3f}")


**Evaluation results (test set, threshold 0.5):**

* **Accuracy:** See results above
* **Sensitivity (TPR):** See results above
* **Specificity (TNR):** See results above


In [ ]:
# Simple variable importance proxy: absolute coefficients
coefs_abs = pd.DataFrame({
    'Feature': X.columns,
    'Abs_Coefficient': np.abs(model.coef_[0])
}).sort_values('Abs_Coefficient', ascending=False)

print("Top coefficients (absolute value):")
print(coefs_abs.head(10))


**Interpretation (coefficients):**
* **TotalSpend** and **AcceptedAny** show strong positive association with response.
* **Higher Recency** (less recent activity) shows a negative association.
* **Income** has a positive effect on response.
* Some **Education** and **Marital_Status** levels differ from their baseline categories.

## Limitations

* **Imbalanced outcome:** if Response=1 is rare (see counts above), sensitivity may be lower at 0.5 threshold.
* **Simple imputation:** median for `Income` may bias variance estimates.
* **Specification:** only linear terms; no interactions or non-linear effects modeled.
* **Causality:** this is predictive/associational, not causal inference.

## Next Steps (for Part 2)

* Try alternative thresholds or ROC/AUC for a fuller view of performance.
* Add **interaction terms** (e.g., `AcceptedAny × TotalSpend`, `Children × Income`) or **nonlinearities** (splines on `Recency`/`Income`).
* Compare with **tree-based** baselines (e.g., random forest) and calibrate probabilities.
* Consider **class balance** methods if Response is rare (class weights or resampling).
